# Грунт против «масложора»

Два участника на одних и тех же данных и одних и тех же тестах:

- **ученик** (`acquire.py` + `core.py`): общие схемы рассуждения без единого слова, язык задач выучивает сам по рассказам с ответами, с 20 рассказов на задачу. Считает на процессоре;
- **обычный трансформер** с нуля (`colab/baseline.py`): тот же текст на входе, ответ на выходе; ему даются и те же 20 рассказов, и в 10–100 раз больше. Считает на видеокарте.

Что проверяется:

1. 20 задач bAbI; 12 новых задач A и 12 задач B (по ним грунт дорабатывался); 12 новых задач C (отложенные: их писали другие авторы, код до них был заморожен и запущен на них один раз).
2. **Без слов и за пределами обучающего размера** (этап 9): 12 задач N и 12 задач M в виде записей из кодов и чисел. Учатся оба на маленьких эпизодах, проверяются на таких же и на в 3 и 10 раз больших.
3. **Свободный язык** (этапы 7–8): 23 задачи, пересказанные другими словами. Ученик на них один даёт 0 %; читалка — маленькая сеть, которую учит ученик и проверяет грунт, — читает их по 20 шаблонным и 40 свободным рассказам. Трансформер получает те же 60 рассказов.

Как запускать:

1. Среда выполнения → Сменить среду → T4 (или L4) и «Большой объём ОЗУ» (там больше процессоров).
2. Среда выполнения → Выполнить всё. Идёт около пяти-шести часов: трансформер учится в фоне, пока считает ученик.
3. Если оборвалось, выполнить всё снова: продолжит с места остановки.
4. Нужен вывод последней ячейки (таблицы между линиями из `=`).

In [ ]:
import os, subprocess, sys
ROOT = os.environ.get('MX_ROOT', '/content')      # где всё лежит
QUICK = bool(os.environ.get('MX_QUICK'))          # короткий прогон каждого шага: только проверить, что всё работает
WORK = ROOT + '/mx-mind'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/mx-mind'
except Exception as e:
    print('Диск не подключён, итог останется в', WORK, '-', e)
os.makedirs(WORK, exist_ok=True)
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
os.environ['PYTHONHASHSEED'] = '0'
!python -u {M5}/colab/mind.py --work {WORK} --fetch
Q = ' --train 60 --test 20' if QUICK else ''     # в коротком прогоне наборы маленькие
for name in 'ABCNM':
    !python {M5}/newtasks.py --set {name} --out {WORK}/new/{name}{Q} | tail -1
for name in 'NM':                                  # записи без слов: те же задачи, но тестовые эпизоды в 3 и 10 раз больше
    for size in (3, 10):
        !python {M5}/newtasks.py --set {name} --size {size} --out {WORK}/new/{name}{size}{Q} | tail -1

## Трансформер (в фоне)

bAbI: те же 20 рассказов на задачу, стандартный набор (1000 вопросов на задачу) и большой (10 000). Новые задачи: 20, 200 и 2000 рассказов на задачу; для записей N и M — ещё и тест на больших эпизодах. Свободный язык: 20 шаблонных и 40 свободных рассказов на задачу, четыре группы задач. Ход пишется в `baseline.txt` в рабочей папке.

In [ ]:
B = sys.executable + ' -u ' + M5 + '/colab/baseline.py' + (' --minutes 0.2' if QUICK else '')
S1, S2 = ('20', '20') if QUICK else ('20,1k,10k', '20,200,2000')
cmd = f"{B} --data {WORK}/tasks_1-20_v1-2 --lang en --sizes {S1} --out {WORK}/baseline_en.json" if not QUICK else 'true'
for name in 'A' if QUICK else 'ABC':
    cmd += f"; {B} --data {WORK}/new/{name} --lang '' --sizes {S2} --out {WORK}/baseline_{name}.json"
for name in 'N' if QUICK else 'NM':
    cmd += f"; {B} --data {WORK}/new/{name} --lang '' --sizes {S2} --sents 150 --also {WORK}/new/{name}3,{WORK}/new/{name}10 --out {WORK}/baseline_{name}.json"
!python {M5}/free/run.py --babi {WORK}/tasks_1-20_v1-2 --new {WORK}/new --out {WORK}/free --plain {WORK}/free_plain --no-reader | tail -1
for g in ('dev',) if QUICK else ('dev', 'held_out_1', 'held_out_2', 'held_out_3'):
    cmd += f"; {B} --data {WORK}/free_plain/{g} --lang '' --sizes 60 --out {WORK}/baseline_free_{g}.json"
base = subprocess.Popen(cmd, shell=True, stdout=open(WORK + '/baseline.txt', 'w'), stderr=subprocess.STDOUT)
print('трансформер запущен в фоне')

## Ученик: новые задачи

Каждая задача учится отдельно и с нуля, с 20 рассказов. Наборы N и M — записи без слов (M — отложенный: написан другими авторами после исправлений по N). Ученик учится на маленьких эпизодах, проверяется ещё и на эпизодах в 3 и 10 раз больше (строки `test … in N3`, `test … in N10`).

In [ ]:
T = '1' if QUICK else '1,2,3,4,5,6,7,8,9,10,11,12'
X = ' --budget 20' if QUICK else ''
for name in 'ABCNM':
    out = f'{WORK}/learner_{name}.txt'
    also = f' --also {WORK}/new/{name}3,{WORK}/new/{name}10' if name in 'NM' else ''
    if not os.path.exists(out) or 'all:' not in open(out).read():
        !python -u {M5}/acquire.py --data {WORK}/new/{name} --stories 20 --apart --tasks {T}{X}{also} > {out}
    print('набор', name); print(''.join(l for l in open(out) if l.startswith(('test', 'all'))))

## Ученик: bAbI

Английский, хинди и английский с заменёнными буквами (шифр) по 20 рассказов на задачу; для английского ещё 10 и 50 рассказов. Раз в пять минут печатает, сколько задач пройдено.

In [ ]:
R = ' --runs en:20 --budget 3' if QUICK else ''
!python -u {M5}/colab/mind.py --work {WORK}{R}

## Свободный язык (этапы 7–8)

Для каждой из 23 задач: ученик учит шаблонный язык по 20 рассказам, читалка учится читать пересказ. В таблице — доля верных ответов на свободном тесте: ученик один, читалка без свободных рассказов, читалка с 40 свободными рассказами с ответами, и лучшее, что вообще допускают её кандидаты. Группы: `dev` — по этим задачам читалка писалась; `held-out 1`, `held-out 2`, `held-out 3` — задачи, отложенные для первого, второго и третьего круга. Считает нынешний код читалки (четвёртый круг), поэтому числа могут быть выше, чем в таблицах README: там прогоны кода, замороженного до того, как задачи были пересказаны.

In [ ]:
O = ' --only b1' if QUICK else ''
!python -u {M5}/free/run.py --babi {WORK}/tasks_1-20_v1-2 --new {WORK}/new --out {WORK}/free --vectors get --jobs {os.cpu_count()}{O} | tee {WORK}/free.txt

## Итог

In [ ]:
try:
    base.wait()
except NameError:
    pass
print('=' * 70)
print('УЧЕНИК, bAbI')
print(open(WORK + '/results.md').read())
for name in 'ABCNM':
    print('УЧЕНИК, новые задачи', name, '(20 рассказов):', [l.strip() for l in open(f'{WORK}/learner_{name}.txt') if l.startswith('all')])
print('УЧЕНИК, записи без слов на больших эпизодах:')
for name in 'NM':
    big = [l.split() for l in open(f'{WORK}/learner_{name}.txt') if l.startswith('test') and ' in ' in l]
    for size in (name + '3', name + '10'):
        rows = [l[4].split('/') for l in big if l[3].rstrip(':') == size]
        if rows:
            print('  ', size, f"{sum(int(r) for r, _ in rows)}/{sum(int(n) for _, n in rows)} = {100 * sum(int(r) for r, _ in rows) / sum(int(n) for _, n in rows):.2f}%")
print('=' * 70)
print('СВОБОДНЫЙ ЯЗЫК: читалка')
print(open(WORK + '/free.txt').read())
print('=' * 70)
print('ТРАНСФОРМЕР (по порядку: bAbI, новые A, B, C, записи N и M с большими эпизодами, свободный язык по группам)')
lines = open(WORK + '/baseline.txt').read().splitlines()
print('\n'.join(l for l in lines if l.startswith('|') or 'test' in l) or '\n'.join(lines[-15:]))
print('=' * 70)